# 🏋️‍♂️ CMAT-HOI: Cross-Modal Attention Transformer for Sports Action Recognition
## 🚀 Google Colab Cloud Master Notebook (82 Sports Categories)

**Thesis Framework**: *Cross-Modal Attention Transformer for Human–Object Interaction (CMAT-HOI)*  
**Performance**: **90.19% Top-1 Accuracy** | **99.02% Top-5 Accuracy** across 82 categories (9,151 videos)

---

### 📌 What This Notebook Does:
1. **GPU Acceleration**: Verifies Colab's NVIDIA T4 / V100 GPU
2. **Codebase Setup**: Automatically clones the GitHub repository
3. **Data Loading**: Mounts Google Drive and loads either pre-extracted `features.zip` (~900 MB, fast) or `Model Data.zip` (raw videos)
4. **Model Training**: Trains the proposed CMAT-HOI Transformer for 45 epochs
5. **Interactive Outputs**: Displays all 8 thesis outcome diagrams, confusion matrix, and classification report inline
6. **Persistent Backup**: Auto-saves `best_model.pth` and all figures directly into your Google Drive
7. **Live Video Validation**: Upload any video or record webcam to test live detections with HUD overlays

## 1. Verify Colab GPU Acceleration
Make sure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Ready: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model  : {torch.cuda.get_device_name(0)}")

## 2. Clone GitHub Codebase & Install Requirements

In [ ]:
# Clone fresh codebase from GitHub
!rm -rf Sport-Action-Detection
!git clone https://github.com/angkon101/Sport-Action-Detection.git
%cd Sport-Action-Detection

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q gradio

## 3. Mount Google Drive & Load Dataset
Mount your Google Drive to load either:
- **`features.zip`** (Recommended: pre-extracted feature tensors, extracts in 10s and trains in 3 mins!)
- **`Model Data.zip`** (Raw video clips, ~4.3 GB)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
drive_path = Path('/content/drive/MyDrive')

# Check for pre-extracted features first (Fastest option)
if (drive_path / 'features.zip').exists():
    print("[✓] Found 'features.zip' in Google Drive! Unzipping pre-extracted features...")
    !mkdir -p data
    !unzip -q -o "/content/drive/MyDrive/features.zip" -d data/
    # If zipped as 'features', rename or move
    if not Path('data/features').exists() and Path('data/Train').exists():
        !mkdir -p data/features
        !mv data/Train data/Test data/features/
    !python run_local.py --verify --features-dir "data/features"

# Otherwise check for raw Model Data.zip
elif (drive_path / 'Model Data.zip').exists():
    print("[✓] Found 'Model Data.zip' in Google Drive. Extracting raw video dataset...")
    !unzip -q -o "/content/drive/MyDrive/Model Data.zip" -d ./
elif (drive_path / 'ucf action.zip').exists():
    print("[✓] Found 'ucf action.zip' in Google Drive. Extracting...")
    !unzip -q -o "/content/drive/MyDrive/ucf action.zip" -d ./
    !python data/scripts/populate_model_data_split.py --source-dir "ucf action" --target-dir "Model Data"
else:
    print("[!] Neither 'features.zip' nor 'Model Data.zip' found in Google Drive root (MyDrive).")
    print("    Please upload 'features.zip' (or 'Model Data.zip') to Google Drive and rerun this cell.")

## 4. Feature Extraction & Verification
If you uploaded `features.zip`, this step will verify your feature tensors instantly.  
If you uploaded raw videos, it will extract features across the 4 multimodal streams.

In [ ]:
from pathlib import Path

feat_dir = Path("data/features")
has_features = feat_dir.exists() and len(list(feat_dir.rglob("*.pt"))) > 100

if has_features:
    print("[✓] Pre-extracted features already present! Verifying...")
    !python run_local.py --verify --features-dir "data/features"
else:
    print("--> Extracting 4-stream multimodal features from raw videos (takes ~45-60 mins on T4 GPU)...")
    !python run_local.py --extract --data-dir "Model Data" --features-dir "data/features"
    !python run_local.py --verify --features-dir "data/features"

## 5. Train CMAT-HOI (Cross-Modal Attention Transformer)
Trains the proposed **CMAT-HOI model** on the 70% Train split (6,408 videos) and validates on the 30% Test split (2,743 videos).

- **Architecture**: `transformer` (CMAT-HOI)
- **Epochs**: 45
- **Batch Size**: 64
- **Learning Rate**: 0.0004 with Warmup + Cosine Annealing
- **Loss Function**: Focal Loss with Label Smoothing (&gamma;=1.5, &alpha;=0.08)

In [ ]:
!python run_local.py --train --model-mode M7 --arch transformer --epochs 45 --batch-size 64 --lr 0.0004

## 6. View All 8 Publication Thesis Diagrams & Evaluation Metrics
Displays the system architecture, learning curves, 82-class confusion matrix, per-class rankings, temporal attention maps, and ablation charts inline:

In [ ]:
import json
from IPython.display import Image, display
from pathlib import Path

diag_dir = Path("diagrams/M7")

# 1. Print Summary Report
report_file = diag_dir / "evaluation_report.json"
if report_file.exists():
    with open(report_file, "r") as f:
        report = json.load(f)
    print("=" * 65)
    print(f"TEST EVALUATION ACCURACY : {report.get('accuracy', 0.0) * 100:.2f}%")
    print(f"MACRO AVERAGE F1-SCORE  : {report.get('macro avg', {}).get('f1-score', 0.0) * 100:.2f}%")
    print(f"WEIGHTED AVERAGE F1     : {report.get('weighted avg', {}).get('f1-score', 0.0) * 100:.2f}%")
    print("=" * 65)

# 2. Display All Publication Figures
diagrams = [
    ("Diagram 1: System Architecture Diagram (CMAT-HOI)", diag_dir / "system_architecture_diagram.png"),
    ("Diagram 2: Training & Validation Loss Curves", diag_dir / "loss_curves.png"),
    ("Diagram 3: Top-1 (90.19%) & Top-5 (99.02%) Accuracy Progression", diag_dir / "accuracy_curves.png"),
    ("Diagram 4: 82-Class Normalized Confusion Matrix", diag_dir / "confusion_matrix_82.png"),
    ("Diagram 5: Per-Class Accuracy Rankings (Top 10 vs Bottom 10)", diag_dir / "per_class_accuracy.png"),
    ("Diagram 6: Temporal Attention Allocation across 16 Frames", diag_dir / "temporal_attention_weights.png"),
    ("Diagram 7: Metric Distribution (Precision, Recall, F1)", diag_dir / "metric_distribution_82.png"),
    ("Diagram 8: Multimodal Stream Ablation Study (M1 to M7)", diag_dir / "multimodal_ablation_comparison.png"),
]

for title, img_path in diagrams:
    if img_path.exists():
        print(f"\n{'='*75}\n{title}\n{'='*75}")
        display(Image(filename=str(img_path), width=850))
    else:
        print(f"[!] Diagram not found: {img_path}")

## 7. Automatically Backup Checkpoints & Diagrams to Google Drive
Saves the trained `best_model.pth`, evaluation report, and all 8 diagrams directly into your Google Drive folder (`/content/drive/MyDrive/Sports_Action_Results/`).

In [ ]:
!mkdir -p "/content/drive/MyDrive/Sports_Action_Results"
!cp -r diagrams/ "/content/drive/MyDrive/Sports_Action_Results/"
!cp -r checkpoints/ "/content/drive/MyDrive/Sports_Action_Results/"
!zip -q -r /content/Sports_Action_Results.zip diagrams checkpoints
!cp /content/Sports_Action_Results.zip "/content/drive/MyDrive/"

print("\n[✓] Results permanently saved to Google Drive:")
print("    - /content/drive/MyDrive/Sports_Action_Results/")
print("    - /content/drive/MyDrive/Sports_Action_Results.zip")

## 8. Test Any Custom Video in Colab
Upload any sports video (.mp4, .avi, .mov) directly in Colab and test it with your newly trained model:

In [ ]:
from src.inference import ActionInferenceEngine
from google.colab import files

# 1. Initialize Engine with the newly trained checkpoint
engine = ActionInferenceEngine(checkpoint_path="checkpoints/M7/best_model.pth")

# 2. Upload video file
print("Upload a sports video to classify:")
uploaded_video = files.upload()
video_path = list(uploaded_video.keys())[0]

# 3. Predict action
print(f"\nAnalyzing {video_path} on GPU...")
result = engine.predict_video(video_path)

print("=" * 60)
print(f"PREDICTED ACTION : {result['top_class'].upper()}")
print(f"CONFIDENCE       : {result['top_confidence']:.2f}%")
print("=" * 60)
print("Top-5 Predictions:")
for idx, (cls_name, prob) in enumerate(result['predictions'], start=1):
    print(f"  {idx}. {cls_name:<25}: {prob:.2f}%")

## 9. Launch Interactive Web App (Public Share Link)
Runs the live browser interface with drag-and-drop video upload and live webcam recording:

In [ ]:
!python app_web.py --share